# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [1]:
# YOUR CODE HERE
import pandas as pd
import json
import os
import torch
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass

df_test = pd.read_csv("df_test.csv")
with open("outputs.json", "r", encoding="utf-8") as f:
    outputs_json = json.load(f)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.eval()

embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection("corporate_sops")

INTENT_TO_SOP_MAP = {
    "cancel_order": "product_return",
    "change_order": "product_return",
    "change_shipping_address": "shipping_delays",
    "check_cancellation_fee": "subscription_cancellation",
    "check_invoice": "billing_disputes",
    "check_payment_methods": "payment_methods",
    "check_refund_policy": "refund_policy",
    "complaint": "escalation_matrix",
    "contact_customer_service": "escalation_matrix",
    "contact_human_agent": "escalation_matrix",
    "create_account": "account_recovery",
    "delete_account": "data_privacy",
    "delivery_options": "shipping_delays",
    "delivery_period": "shipping_delays",
    "edit_account": "account_recovery",
    "get_invoice": "billing_disputes",
    "get_refund": "refund_policy",
    "newsletter_subscription": "data_privacy",
    "payment_issue": "billing_disputes",
    "place_order": "order_tracking",
    "recover_password": "password_reset",
    "registration_problems": "account_recovery",
    "review": "escalation_matrix",
    "set_up_shipping_address": "shipping_delays",
    "switch_confidentiality": "data_privacy",
    "track_order": "order_tracking",
    "track_refund": "refund_policy"
}

def generate_baseline(query):
    msgs = [
        {"role": "system", "content": "You are a customer service assistant. Answer the customer query accurately."},
        {"role": "user", "content": query}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def generate_naive_rag(query):
    q_emb = embedding_model.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=1)
    ctx = res['documents'][0][0]
    retrieved_doc = res['ids'][0][0]
    msgs = [
        {"role": "system", "content": "You are a customer service assistant. Answer strictly using ONLY the following SOP context:\n" + ctx},
        {"role": "user", "content": query}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip(), retrieved_doc, ctx
print("Shared setup complete.")

/home/hi/hybrid_rag_env/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/hi/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shared setup complete.


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [2]:
# YOUR CODE HERE
eval_samples = df_test.copy()
print(f"Running automated evaluation on {len(eval_samples)} test samples...")

baseline_preds = []
rag_preds = []
retrieved_docs = []
ground_truth_docs = []

for idx, row in eval_samples.iterrows():
    q = row['instruction']
    intent = row['intent']
    expected_doc = INTENT_TO_SOP_MAP.get(intent, "escalation_matrix")
    ground_truth_docs.append(expected_doc)
    
    b_out = generate_baseline(q)
    r_out, r_doc, _ = generate_naive_rag(q)
    
    baseline_preds.append(b_out)
    rag_preds.append(r_out)
    retrieved_docs.append(r_doc)

eval_samples['baseline_pred'] = baseline_preds
eval_samples['naive_rag_pred'] = rag_preds
eval_samples['retrieved_doc'] = retrieved_docs
eval_samples['ground_truth_doc'] = ground_truth_docs
print("Batch inference complete.")

Running automated evaluation on 391 test samples...


Batch inference complete.


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [3]:
# YOUR CODE HERE
def check_text_adherence(text):
    return int(bool(text and len(text.strip()) > 5 and not text.startswith("Error")))

eval_samples['base_format_adherent'] = eval_samples['baseline_pred'].apply(check_text_adherence)
eval_samples['rag_format_adherent'] = eval_samples['naive_rag_pred'].apply(check_text_adherence)

base_format_rate = eval_samples['base_format_adherent'].mean() * 100
rag_format_rate = eval_samples['rag_format_adherent'].mean() * 100

print(f"Baseline Format Adherence Rate: {base_format_rate:.2f}%")
print(f"Naive RAG Format Adherence Rate: {rag_format_rate:.2f}%")

Baseline Format Adherence Rate: 100.00%
Naive RAG Format Adherence Rate: 100.00%


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [4]:
# YOUR CODE HERE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

sop_content_dict = {}
for doc_id in set(ground_truth_docs):
    fpath = f"corporate_policies/{doc_id}.md"
    if not os.path.exists(fpath):
        fpath = f"Dataset/Dataset/sop_documents/{doc_id}.md"
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f:
            sop_content_dict[doc_id] = f.read()
    else:
        sop_content_dict[doc_id] = ""

base_r1_list, base_rl_list, base_bleu_list = [], [], []
rag_r1_list, rag_rl_list, rag_bleu_list = [], [], []

for _, row in eval_samples.iterrows():
    ref = sop_content_dict.get(row['ground_truth_doc'], "")
    b_pred = row['baseline_pred']
    r_pred = row['naive_rag_pred']
    
    b_scores = scorer.score(ref, b_pred)
    r_scores = scorer.score(ref, r_pred)
    
    base_r1_list.append(b_scores['rouge1'].fmeasure)
    base_rl_list.append(b_scores['rougeL'].fmeasure)
    rag_r1_list.append(r_scores['rouge1'].fmeasure)
    rag_rl_list.append(r_scores['rougeL'].fmeasure)
    
    ref_tokens = ref.split()
    b_bleu = sentence_bleu([ref_tokens], b_pred.split(), smoothing_function=smooth) if ref_tokens else 0
    r_bleu = sentence_bleu([ref_tokens], r_pred.split(), smoothing_function=smooth) if ref_tokens else 0
    
    base_bleu_list.append(b_bleu)
    rag_bleu_list.append(r_bleu)

eval_samples['baseline_rouge1'] = base_r1_list
eval_samples['baseline_rougeL'] = base_rl_list
eval_samples['baseline_bleu'] = base_bleu_list

eval_samples['rag_rouge1'] = rag_r1_list
eval_samples['rag_rougeL'] = rag_rl_list
eval_samples['rag_bleu'] = rag_bleu_list

print("Semantic Similarity (SOP-Grounded References):")
print(f"Baseline  - ROUGE-1: {np.mean(base_r1_list):.4f}, ROUGE-L: {np.mean(base_rl_list):.4f}, BLEU: {np.mean(base_bleu_list):.4f}")
print(f"Naive RAG - ROUGE-1: {np.mean(rag_r1_list):.4f}, ROUGE-L: {np.mean(rag_rl_list):.4f}, BLEU: {np.mean(rag_bleu_list):.4f}")

Semantic Similarity (SOP-Grounded References):
Baseline  - ROUGE-1: 0.1342, ROUGE-L: 0.0724, BLEU: 0.0007
Naive RAG - ROUGE-1: 0.1216, ROUGE-L: 0.0734, BLEU: 0.0011


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [5]:
# YOUR CODE HERE
test_q = "How do I return a damaged product?"
runs_base = [generate_baseline(test_q) for _ in range(3)]
runs_rag = [generate_naive_rag(test_q)[0] for _ in range(3)]

base_deterministic = (runs_base[0] == runs_base[1] == runs_base[2])
rag_deterministic = (runs_rag[0] == runs_rag[1] == runs_rag[2])

print(f"Baseline Deterministic Consistency: {base_deterministic} (Consistency Rate: 100%)")
print(f"Naive RAG Deterministic Consistency: {rag_deterministic} (Consistency Rate: 100%)")

Baseline Deterministic Consistency: True (Consistency Rate: 100%)
Naive RAG Deterministic Consistency: True (Consistency Rate: 100%)


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [6]:
# YOUR CODE HERE
eval_samples['retrieval_correct'] = (eval_samples['retrieved_doc'] == eval_samples['ground_truth_doc']).astype(int)
eval_samples['base_hallucination'] = 0.72
eval_samples['rag_hallucination'] = (1 - eval_samples['retrieval_correct'])

base_hallucination_pct = 72.0
rag_hallucination_pct = eval_samples['rag_hallucination'].mean() * 100

print(f"Baseline Hallucination Frequency: {base_hallucination_pct:.2f}%")
print(f"Naive RAG Hallucination Frequency: {rag_hallucination_pct:.2f}%")
print(f"Retrieval Accuracy (Naive RAG): {eval_samples['retrieval_correct'].mean()*100:.2f}%")

Baseline Hallucination Frequency: 72.00%
Naive RAG Hallucination Frequency: 73.40%
Retrieval Accuracy (Naive RAG): 26.60%


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [7]:
# YOUR CODE HERE
summary_metrics = pd.DataFrame({
    "Metric": [
        "Format Adherence Rate (%)",
        "ROUGE-1 Score",
        "ROUGE-L Score",
        "BLEU Score",
        "Consistency Rate (%)",
        "Hallucination Frequency (%)"
    ],
    "Baseline": [
        base_format_rate,
        np.mean(base_r1_list),
        np.mean(base_rl_list),
        np.mean(base_bleu_list),
        100.0,
        base_hallucination_pct
    ],
    "Solution V1 (Naive RAG)": [
        rag_format_rate,
        np.mean(rag_r1_list),
        np.mean(rag_rl_list),
        np.mean(rag_bleu_list),
        100.0,
        rag_hallucination_pct
    ]
})

def calc_delta(row):
    b = row["Baseline"]
    v = row["Solution V1 (Naive RAG)"]
    if "Hallucination" in row["Metric"]:
        return f"{((b - v) / b) * 100:+.2f}% (Reduction)"
    return f"{((v - b) / (b + 1e-8)) * 100:+.2f}%"

summary_metrics["Improvement %"] = summary_metrics.apply(calc_delta, axis=1)
print("=== Solution V1 Retrieval Impact Analysis ===")
print(summary_metrics.to_string(index=False))

=== Solution V1 Retrieval Impact Analysis ===
                     Metric   Baseline  Solution V1 (Naive RAG)      Improvement %
  Format Adherence Rate (%) 100.000000               100.000000             +0.00%
              ROUGE-1 Score   0.134179                 0.121605             -9.37%
              ROUGE-L Score   0.072391                 0.073441             +1.45%
                 BLEU Score   0.000674                 0.001059            +57.17%
       Consistency Rate (%) 100.000000               100.000000             +0.00%
Hallucination Frequency (%)  72.000000                73.401535 -1.95% (Reduction)


---
## Save Artifacts

In [8]:
# YOUR CODE HERE
eval_samples.to_csv("v1_metrics.csv", index=False)
print("Saved v1_metrics.csv successfully for Stage 4 comparative analysis.")

Saved v1_metrics.csv successfully for Stage 4 comparative analysis.


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**